# AdaptiveGPT on Kaggle

Trains the three stages, runs the four-arm comparison, writes the figures.

**Before running anything:**

1. Accelerator -> **GPU T4 x2**. Not P100 - it is Pascal (sm_60) and current PyTorch
   wheels ship no kernels below sm_70, so every CUDA op dies rather than running slowly.
2. Internet -> **On** (needed for pip; requires phone verification on Kaggle).
3. Attach your tokenised corpus as a Dataset - the three `*_tokens.npy` files. The
   config cell finds wherever Kaggle mounted it and prints the path; you do not need to
   guess it (a real mount looked like
   `/kaggle/input/datasets/<user>/<slug>/data/wikitext103`).

**Budget.** Stages 1-3 are 3600 + 1500 + 3000 steps. Measure step time in the preflight
cell before assuming it fits the 12 h session cap; if it does not, run Stage 1 alone and
resume with `--resume latest`.

In [ ]:
# Re-runnable: a second run of this cell would otherwise fail with
# "destination path already exists and is not an empty directory".
# Only the checkout is removed -- runs and results live outside it.
import shutil, os
REPO_DIR = "/kaggle/working/agpt"
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone -q -b amt-dev https://github.com/cjaitej/MemAdapt.git {REPO_DIR}
%cd {REPO_DIR}
!pip install -q tiktoken
!python -m pytest -q

## Config

`MAX_STEPS_1 = 3600` is **two epochs**: WikiText-103 is 117,690,368 training tokens, so
one epoch is 1,795 steps at 65536 tokens/step.

`EVAL_STEPS = 40` is capped by the validation split, which holds only 245,760 usable
tokens = 60 batches at B=8/T=512. Above 60 you are re-measuring the same data.

`DROPOUT = 0.1` because training here is multi-epoch by construction. The nanoGPT
default of 0.0 assumes a single pass over something far larger.

In [ ]:
# Kaggle mounts a dataset at a path built from its slug and internal folder layout,
# which is almost never what you would type from memory. Locate it once, here, and
# pass it explicitly from then on -- the trainer records --data-dir in config.json,
# and "whatever the auto-discovery happened to find" is not a reproducible answer.
from agpt.data.loaders import find_shard_dirs

DATA = ""                                  # set this if you already know the path
if not DATA:
    found = find_shard_dirs()
    assert found, "no *_tokens.npy anywhere under /kaggle/input -- attach the dataset"
    assert len(found) == 1, f"several corpora found, pick one: {found}"
    DATA = found[0]
print("DATA =", DATA)

RUNS    = "/kaggle/working/runs"
RESULTS = "/kaggle/working/results"
FIGURES = "/kaggle/working/figures"

BATCH        = 16          # must divide 128 (total-batch-tokens / block-size)
DROPOUT      = 0.1
EVAL_STEPS   = 40
CKPT_EVERY   = 500
MAX_STEPS_1  = 3600        # stage 1, dense -- 2 epochs
MAX_STEPS_2  = 1500        # stage 2, routers
MAX_STEPS_3  = 3000        # stage 3, joint
LAMBDA_DEPTH = 0.05        # the quality/compute knob; sweep it later

COMMON = (f"--data-dir {DATA} --out-dir {RUNS} --batch-size {BATCH} "
          f"--dropout {DROPOUT} --eval-steps {EVAL_STEPS} "
          f"--ckpt-every {CKPT_EVERY} --compile")
print(COMMON)

## Preflight

Two things this settles before quota is spent:

- **What the card is.** The banner must report `bf16=False` on a T4, and the trainer
  must then choose `fp16 + GradScaler`. fp16 without a scaler trains quietly worse
  rather than failing, so the pairing is made for you - just confirm it happened.
- **Whether routing is worth anything here.** Routing removes arithmetic but not kernel
  launches, and adds a gather and a scatter on top. Measured in eager mode on an RTX
  3050, the adaptive arm ran at **0.29x dense** while nothing was exiting. `--compile`
  is what closes that gap. If the `adaptive` row comes out below 1.0x here, that is a
  result to report, not a setup problem to work around.

Three arms get compiled, so allow ~15 minutes.

In [ ]:
!python -c "from agpt.precision import describe_device; print(describe_device())"

cmd = f"python scripts/benchmark.py --compile --batch-sizes 8 16 32 --out {RESULTS}/benchmark.json"
print(cmd)
!{cmd}

## Stage 1 - the dense language model

Routers exist but every gate is pinned open, so this is exactly a dense model. Its
checkpoint becomes **both** the dense baseline and the Stage 2 initialisation, which is
what removes seed variance from the headline comparison.

Add `--resume latest` with the same `--run-name` to continue after a session timeout.

In [ ]:
cmd = f"python -m agpt.train --stage dense --run-name s1_dense --max-steps {MAX_STEPS_1} {COMMON}"
print(cmd)
!{cmd}

## Stage 2 - fit the routers

Backbone frozen; routers trained by BCE against convergence labels derived from a dense
forward pass, which costs one extra forward per step.

Watch `depth` in the log. It starts at 12.0 - the routers initialise to "always
continue" so that a warm start is never damaged at step 0 - and should fall as the depth
penalty ramps in over the first 20% of steps.

In [ ]:
cmd = (f"python -m agpt.train --stage routers --run-name s2_routers "
       f"--init-from {RUNS}/s1_dense/best.pt --max-steps {MAX_STEPS_2} {COMMON}")
print(cmd)
!{cmd}

## Stage 3 - joint fine-tune

Everything unfreezes and the model adapts to being interrupted.

**`best.pt` is the wrong checkpoint to take from this stage.** It selects on validation
loss, and under depth pressure the best loss occurs at step 0, before any token exits.
Take the final `ckpt_*.pt` instead.

In [ ]:
cmd = (f"python -m agpt.train --stage joint --run-name s3_joint "
       f"--init-from {RUNS}/s2_routers/best.pt --lambda-depth {LAMBDA_DEPTH} "
       f"--max-steps {MAX_STEPS_3} {COMMON}")
print(cmd)
!{cmd}

In [ ]:
import glob
S3 = sorted(glob.glob(f"{RUNS}/s3_joint/ckpt_*.pt"))[-1]   # not best.pt -- see above
print("using", S3)

## The four-arm comparison

All four arms are built from this one checkpoint by swapping `exit_mode`, so they differ
in the routing rule and nothing else - not the seed, not the token budget, not the data
order. `random` and `fixed` are matched automatically to the depth the adaptive arm
actually reached; an unmatched baseline settles nothing.

How to read it: beat **random** or the router only learned to hit a budget; beat
**fixed** or per-token adaptivity buys nothing over a uniformly shallower model.

In [ ]:
cmd = (f"python scripts/compare.py --ckpt {S3} --data-dir {DATA} "
       f"--batch-size {BATCH} --eval-steps {EVAL_STEPS} --out {RESULTS}/compare.json")
print(cmd)
!{cmd}

In [ ]:
cmd = (f"python scripts/evaluate.py --ckpt {S3} --data-dir {DATA} "
       f"--batch-size {BATCH} --eval-steps {EVAL_STEPS} "
       f"--confidence --oracle --gen-tokens 100 500 --out {RESULTS}/evaluate.json")
print(cmd)
!{cmd}

`--oracle` is the diagnostic to read if the numbers disappoint:

- router agrees with the label but quality is bad -> the **rule** is wrong; try
  `--target-type kl`
- router disagrees and the oracle depth is shallow -> the router failed to learn a
  signal that was there
- the oracle depth is itself deep -> there is no easy-token structure here to exploit,
  and no router can invent it. That is a real finding about the model and the corpus.

In [ ]:
cmd = (f"python scripts/figures.py --compare {RESULTS}/compare.json "
       f"--benchmark {RESULTS}/benchmark.json "
       f"--confidence {RESULTS}/confidence.json --out-dir {FIGURES}")
print(cmd)
!{cmd}

from IPython.display import Image, display
import glob
for p in sorted(glob.glob(f"{FIGURES}/*.png")):
    print(p)
    display(Image(p))

## Optional: the depth-penalty sweep (uses both cards)

`--lambda-depth` is the only knob that moves the quality/compute trade, so this sweep
*is* the Pareto figure. One point is not a curve.

One process per card. DDP would make a single run finish sooner; this finishes *more*
runs in the same wall clock, which is what a sweep needs. Kaggle bills the session
rather than the card, so the second GPU costs nothing.

In [ ]:
cmd = (f"python scripts/sweep.py --lambdas 0.01 0.03 0.05 0.1 0.2 "
       f"--init-from {RUNS}/s2_routers/best.pt --data-dir {DATA} "
       f"--out-dir {RUNS}/sweep --max-steps {MAX_STEPS_3} --batch-size {BATCH} "
       f"-- --compile --dropout {DROPOUT} --eval-steps {EVAL_STEPS}")
print(cmd)
!{cmd}

## Save what matters

`/kaggle/working` persists when you commit the notebook, but checkpoints are large.
Keep the JSON, the figures, the training logs, and the one checkpoint you need.

In [ ]:
import glob, os, shutil
os.makedirs(f"/kaggle/working/keep", exist_ok=True)
for p in glob.glob(f"{RESULTS}/*.json") + glob.glob(f"{FIGURES}/*"):
    shutil.copy(p, f"/kaggle/working/keep/")
for run in ("s1_dense", "s2_routers", "s3_joint"):
    for name in ("config.json", "log.jsonl"):
        src = f"{RUNS}/{run}/{name}"
        if os.path.exists(src):
            shutil.copy(src, f"/kaggle/working/keep/{run}_{name}")
shutil.copy(S3, f"/kaggle/working/keep/s3_joint_final.pt")
print(sorted(os.listdir(f"/kaggle/working/keep")))